# Latihan SKLearn: klasifikasi diabetes, SVR, dan SVM kernel linear

Notebook ini menggabungkan tiga latihan:

1. **Klasifikasi diabetes**: memuat data, membagi dataset, standarisasi fitur, melatih model, dan evaluasi.
2. **Support Vector Regression (SVR)**: memprediksi gaji dari lama pengalaman kerja.
3. **SVM kernel linear**: klasifikasi kelulusan ujian dengan perhitungan manual dan Scikit-learn.

Setiap bagian memuat datanya sendiri dan bisa dijalankan terpisah, selama berkas CSV untuk bagian 1 dan 2 tersedia.

# Bagian 1: klasifikasi diabetes

Bagian ini mengikuti lima tahap: memuat data ke DataFrame, membagi dataset, menstandarisasi fitur, melatih model, dan mengevaluasi hasil. Label `Outcome` bernilai `1` untuk positif diabetes dan `0` untuk negatif.

## 1.1 Memuat data ke DataFrame

In [ ]:
import pandas as pd

# Ubah path jika diabetes.csv berada di folder lain.
file_path = r'C:\Users\user\Downloads\diabetes.csv'
df = pd.read_csv(file_path)

display(df.head())
print(f'Ukuran dataset: {df.shape}')
print('Kolom:', df.columns.tolist())
display(df['Outcome'].value_counts().rename('Jumlah'))

Dataset diharapkan memiliki delapan kolom fitur dan satu kolom target `Outcome`.

## 1.2 Membagi dataset

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='Outcome')
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Jumlah data latih: {len(X_train)}')
print(f'Jumlah data uji: {len(X_test)}')

`stratify=y` menjaga proporsi kelas 0 dan 1 tetap sebanding pada data latih dan data uji.

## 1.3 Standarisasi fitur

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Scaler dipasang hanya pada data latih, lalu diterapkan ke data uji agar informasi data uji tidak bocor ke proses pelatihan.

## 1.4 Membuat dan melatih model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print('Model Logistic Regression selesai dilatih.')

## 1.5 Evaluasi model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = model.predict(X_test_scaled)
print(f'Akurasi: {accuracy_score(y_test, y_pred):.3f}')
print('\nLaporan klasifikasi:')
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=['Tidak diabetes (0)', 'Diabetes (1)'],
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=['Tidak diabetes (0)', 'Diabetes (1)'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Data Uji')
plt.tight_layout()
plt.show()

Akurasi menunjukkan proporsi prediksi yang benar. Periksa juga recall kelas `1` untuk melihat berapa banyak pasien positif diabetes yang berhasil dikenali model, dan confusion matrix untuk jumlah prediksi benar dan salah per kelas.

# Bagian 2: Support Vector Regression (SVR)

Tahapan latihan:

1. Mengubah data menjadi DataFrame.
2. Memisahkan atribut dan label.
3. Melatih model SVR.
4. Membuat plot dari model.

Bagian ini membaca `Salary_Data.csv` dari folder kerja notebook.

## 2.1 Mengubah data menjadi DataFrame

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataset = pd.read_csv('Salary_Data.csv')
df = pd.DataFrame(dataset)

display(df.head())
print('Jumlah data:', len(df))

## 2.2 Memisahkan atribut dan label

In [ ]:
X = df[['YearsExperience']]
y = df['Salary']

print('Atribut (X):')
display(X.head())

print('Label (y):')
display(y.head())

## 2.3 Melatih model SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor

model = make_pipeline(
    StandardScaler(),
    SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
)

# Target ikut distandarisasi agar epsilon=0.1 bermakna pada skala yang sama.
svr_model = TransformedTargetRegressor(
    regressor=model,
    transformer=StandardScaler()
)

svr_model.fit(X, y)
print('Model SVR berhasil dilatih!')

## 2.4 Membuat plot dari model

In [ ]:
# 300 titik merata supaya kurva tergambar halus.
X_plot = pd.DataFrame(
    np.linspace(X['YearsExperience'].min(), X['YearsExperience'].max(), 300),
    columns=['YearsExperience']
)

y_plot = svr_model.predict(X_plot)

plt.figure(figsize=(9, 5))
plt.scatter(X['YearsExperience'], y, label='Data aktual')
plt.plot(X_plot['YearsExperience'], y_plot, label='Prediksi SVR')
plt.title('Regresi Support Vector Regression (SVR)')
plt.xlabel('Years of Experience')
plt.ylabel('Salary')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Model SVR dilatih dengan atribut `YearsExperience` untuk memprediksi label `Salary`. Plot membandingkan data aktual dengan kurva prediksi model.

# Bagian 3: SVM kernel linear

Bagian ini mengerjakan klasifikasi ujian berdasarkan dua fitur:

- `x1`: lama belajar
- `x2`: lama tidur
- `y = +1`: lulus
- `y = -1`: tidak lulus

Isinya perhitungan manual, pelatihan model SVM linear dengan Scikit-learn, dan visualisasi decision boundary serta margin.

## 3.1 Import library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm

%matplotlib inline

## 3.2 Perhitungan manual SVM

Empat support vector yang dipakai pada contoh perhitungan:

| Titik | Kelas |
|---|---:|
| (3, 7) | +1 |
| (4, 6) | +1 |
| (1, 6) | -1 |
| (2, 5) | -1 |

Persamaannya:

- `3w1 + 7w2 + b = 1`
- `4w1 + 6w2 + b = 1`
- `w1 + 6w2 + b = -1`
- `2w1 + 5w2 + b = -1`

Pengurangan persamaan dalam masing-masing kelas menghasilkan `w1 = w2`. Dari situ diperoleh `w1 = 2/3`, `w2 = 2/3`, dan `b = -17/3`.

In [ ]:
w1 = 2 / 3
w2 = 2 / 3
b_manual = -17 / 3

print(f"w1 = {w1:.6f}")
print(f"w2 = {w2:.6f}")
print(f"b  = {b_manual:.6f}")

## 3.3 Persamaan decision boundary dan margin

Decision boundary:
\[
\frac{2}{3}x_1 + \frac{2}{3}x_2 - \frac{17}{3} = 0
\]
atau:
\[
x_2 = -x_1 + 8.5
\]

Garis margin diperoleh dari nilai fungsi keputusan `+1` dan `-1`:

- Margin positif: `x2 = -x1 + 10`
- Margin negatif: `x2 = -x1 + 7`

**Catatan:** nilai margin ini dihitung langsung dari parameter `w` dan `b` di atas, jadi bukan `-x1 + 9` dan `-x1 + 8`.

In [ ]:
x1_range = np.linspace(0, 6, 200)

decision_manual = -x1_range + 8.5
margin_positive_manual = -x1_range + 10
margin_negative_manual = -x1_range + 7

plt.figure(figsize=(8, 6))
plt.plot(x1_range, decision_manual, label="Decision boundary")
plt.plot(x1_range, margin_positive_manual, "--", label="Margin positif (+1)")
plt.plot(x1_range, margin_negative_manual, "--", label="Margin negatif (-1)")
plt.xlim(0, 6)
plt.ylim(0, 10)
plt.xlabel("Lama belajar (x1)")
plt.ylabel("Lama tidur (x2)")
plt.title("Persamaan SVM hasil perhitungan manual")
plt.grid(True)
plt.legend()
plt.show()

## 3.4 Menyiapkan dataset

Dataset ini sama dengan dataset klasifikasi ujian pada materi.

In [ ]:
x1 = np.array([2, 1, 3, 1, 4, 2, 5, 2])  # Study Hours
x2 = np.array([8, 6, 7, 4, 6, 5, 7, 3])  # Sleep Hours
y = np.array([1, -1, 1, -1, 1, -1, 1, -1])  # Passed Exam

X = np.column_stack((x1, x2))

print("Bentuk X:", X.shape)
print("Bentuk y:", y.shape)
print("\nData X:")
print(X)
print("\nLabel y:")
print(y)

## 3.5 Membuat dan melatih model SVM linear

Model memakai `SVC` dengan `kernel='linear'`. Nilai `C=1000` membuat penalti terhadap kesalahan klasifikasi besar, sesuai contoh pemisahan linear pada dataset ini.

In [ ]:
clf = svm.SVC(kernel="linear", C=1000)
clf.fit(X, y)

w = clf.coef_[0]
b = clf.intercept_[0]

print("Bobot model:")
print("w1 =", w[0])
print("w2 =", w[1])
print("Bias b =", b)

print("\nSupport vectors:")
print(clf.support_vectors_)

## 3.6 Visualisasi decision boundary dan margin dari model

Pada model linear, decision boundary memenuhi `w1*x1 + w2*x2 + b = 0`. Garis margin memenuhi nilai fungsi keputusan `+1` dan `-1`.

In [ ]:
x1_range = np.linspace(0, 6, 200)

decision_boundary = -(w[0] * x1_range + b) / w[1]
margin_positive = -(w[0] * x1_range + b - 1) / w[1]
margin_negative = -(w[0] * x1_range + b + 1) / w[1]

plt.figure(figsize=(9, 7))

plt.scatter(
    x1[y == 1], x2[y == 1],
    color="blue", label="Lulus (+1)", s=100
)
plt.scatter(
    x1[y == -1], x2[y == -1],
    color="red", label="Tidak lulus (-1)", s=100
)

plt.scatter(
    clf.support_vectors_[:, 0],
    clf.support_vectors_[:, 1],
    s=220, facecolors="none", edgecolors="black",
    linewidths=1.5, label="Support vectors"
)

plt.plot(x1_range, decision_boundary, color="green",
         linewidth=2, label="Decision boundary")
plt.plot(x1_range, margin_positive, "--", color="orange",
         label="Margin positif (+1)")
plt.plot(x1_range, margin_negative, "--", color="purple",
         label="Margin negatif (-1)")

plt.title("SVM Linear - Klasifikasi Ujian")
plt.xlabel("Lama belajar (x1)")
plt.ylabel("Lama tidur (x2)")
plt.xlim(0, 6)
plt.ylim(0, 10)
plt.grid(True)
plt.legend()
plt.show()

## 3.7 Membandingkan hasil manual dan Scikit-learn

Selisih kecil pada angka bisa muncul dari proses optimasi model. Bandingkan parameter yang dicetak model dengan hasil manual.

In [ ]:
print("Hasil perhitungan manual:")
print(f"w1 = {w1:.6f}, w2 = {w2:.6f}, b = {b_manual:.6f}")

print("\nHasil Scikit-learn:")
print(f"w1 = {w[0]:.6f}, w2 = {w[1]:.6f}, b = {b:.6f}")

print("\nPersamaan decision boundary dari model:")
print(f"x2 = -({w[0]:.6f} * x1 + {b:.6f}) / {w[1]:.6f}")

## 3.8 Prediksi data baru

Masukkan dua fitur (lama belajar dan lama tidur) untuk melihat kelas yang diprediksi model.

In [ ]:
# 3 jam belajar dan 7 jam tidur
data_baru = np.array([[3, 7]])

prediksi = clf.predict(data_baru)
print("Data baru:", data_baru[0])
print("Prediksi kelas:", prediksi[0])
print("Keterangan:", "Lulus" if prediksi[0] == 1 else "Tidak lulus")

## 3.9 Kesimpulan

Perhitungan manual menghasilkan:

- `w1 = 2/3`
- `w2 = 2/3`
- `b = -17/3`
- Decision boundary: `x2 = -x1 + 8.5`
- Margin positif: `x2 = -x1 + 10`
- Margin negatif: `x2 = -x1 + 7`

Scikit-learn melatih model dari dataset dan menampilkan support vectors, decision boundary, serta margin.